# Exploratory Data Analysis (EDA) - Titanic Dataset

This notebook performs an end-to-end Exploratory Data Analysis on the classic Titanic dataset. The goal is to examine the structure of the data, identify patterns, clean missing values, spot anomalies, and generate an automated profiling report.

## 1. Setup & Imports
Installing and importing the required libraries for our analysis.

In [ ]:
# Run this cell to install required libraries if you are in a fresh environment
!pip install pandas numpy ydata-profiling

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading
We will load the classic Titanic dataset from a public raw GitHub URL so this notebook is fully reproducible without manual downloads.

In [ ]:
# Load the classic Titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Display the first few rows to ensure it loaded correctly
df.head()

## 3. Manual EDA & Data Cleaning
### Feature Understanding
First, let's understand the shape, data types, and basic statistics of our dataset.

In [ ]:
print("--- Dataset Info ---")
df.info()

print("\n--- Basic Summary Statistics (Numerical) ---")
display(df.describe())

print("\n--- Categorical Features Statistics ---")
display(df.describe(include=['O']))

### Handling Missing Values
Let's identify columns with missing data and apply appropriate imputation or deletion strategies.

In [ ]:
# Check for missing values
print("Missing values per column before cleaning:\n", df.isnull().sum())

# 1. Age: Impute missing ages with the median age to avoid outlier influence.
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Embarked: Impute the 2 missing values with the mode (most frequent port).
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3. Cabin: Has too many missing values (~77%), so we will drop this column entirely.
# We will also drop 'Ticket' and 'PassengerId' as they are random identifiers and not immediately useful for basic statistical patterns.
df.drop(columns=['Cabin', 'Ticket', 'PassengerId'], inplace=True)

# Verify missing values are handled
print("\nMissing values after cleaning:\n", df.isnull().sum())

### Spotting Anomalies
We need to ensure there are no impossible values, such as negative ages or fares.

In [ ]:
# Check for anomalies like Age < 0 or Fare < 0
print("Number of passengers with negative Age:", (df['Age'] < 0).sum())
print("Number of passengers with negative Fare:", (df['Fare'] < 0).sum())

# Some fares are 0, which might be an anomaly or free tickets for staff/infants. 
print("Number of passengers with zero Fare:", (df['Fare'] == 0).sum())

# We'll replace zero fares with the median fare of their respective Pclass
for pclass in [1, 2, 3]:
    median_fare = df[df['Pclass'] == pclass]['Fare'].median()
    # Using boolean indexing to update the values
    df.loc[(df['Fare'] == 0) & (df['Pclass'] == pclass), 'Fare'] = median_fare
    
print("Anomalies addressed. Data is ready for profiling.")

## 4. Automated Profiling
We use `ydata-profiling` to automatically generate a rich, interactive HTML report covering distributions, correlations, missing values, and warnings.

In [ ]:
from ydata_profiling import ProfileReport
ProfileReport(df).to_file("report.html")

print("\nAutomated profiling complete! Please open 'report.html' in your browser to view the interactive EDA.")

## 5. Insights Summary

Based on the manual EDA steps and the statistical distributions observed in the Titanic dataset, we can derive the following key insights:

*   **Overall Survival Rate:** Only about 38% of passengers in the dataset survived the disaster, highlighting the severe fatality rate of the event.
*   **Gender Disparity in Survival:** Females had a significantly higher survival rate (approx. 74%) compared to males (approx. 19%), indicating that the "women and children first" protocol was strictly enforced.
*   **Socioeconomic Status (Class) Matters:** Passengers in 1st class had a much higher chance of survival (over 60%) than those in 3rd class (around 24%), showing that socioeconomic status played a major role in accessing lifeboats.
*   **The Age Factor:** Younger passengers, particularly young children, had better survival odds, whereas elderly passengers had the lowest survival rates overall.
*   **Family Size Influence:** Passengers traveling with small families (1-3 members) survived at higher rates than those traveling completely alone or with very large families.
*   **Fare Correlation:** Higher ticket fares strongly correlate with survival, which inherently links back to the high survival rate of 1st-class passengers who paid premium fares.
*   **Embarkation Port:** Passengers who boarded at Cherbourg (C) had a noticeably higher survival rate compared to those who boarded at Southampton (S) or Queenstown (Q), likely driven by a larger proportion of 1st-class passengers boarding at Cherbourg.